In [3]:
%pip install requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [4]:
from __future__ import annotations

import json
import time
from dataclasses import dataclass, asdict
from datetime import date
from pathlib import Path
from typing import Iterable, Optional

import requests
from bs4 import BeautifulSoup


@dataclass
class FilingMetadata:
    ticker: str
    company_name: str
    cik: str
    form: str
    filing_date: str
    report_date: str
    accession_number: str
    primary_document: str
    primary_url: str
    filing_index_url: str
    raw_submission_url: str
    local_html_path: Optional[str] = None
    local_txt_path: Optional[str] = None


D:\conda_envs\finder\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [5]:
class SECDownloader:
    """
    Download SEC EDGAR 10-K / 10-Q filings.

    Features
    --------
    - Resolve ticker -> CIK
    - Retrieve SEC submissions metadata
    - Handle older filing-history JSON files
    - Filter by form/date
    - Download primary HTML document
    - Optionally download complete submission .txt
    - Extract clean text from HTML
    - Save metadata JSON
    - Skip files already downloaded
    - Basic SEC rate limiting
    - Retry transient HTTP errors

    Example
    -------
    downloader = SECDownloader(
        user_agent="Your Name your_email@example.com"
    )

    filings = downloader.download_company(
        ticker="AAPL",
        start_date="2020-01-01",
        end_date="2026-12-31",
    )
    """

    SEC_DATA_URL = "https://data.sec.gov"
    SEC_ARCHIVES_URL = "https://www.sec.gov/Archives/edgar/data"
    DEFAULT_FORMS = ("10-K", "10-Q")

    def __init__(
        self,
        user_agent: str,
        output_dir: str = "./sec_filings",
        request_delay: float = 0.15,
        timeout: int = 30,
        max_retries: int = 3,
    ):
        if not user_agent or "@" not in user_agent:
            raise ValueError(
                "Please provide a descriptive SEC User-Agent, "
                "e.g. 'Your Name your_email@example.com'"
            )
        self.user_agent = user_agent
        self.output_dir = Path(output_dir)
        self.request_delay = request_delay
        self.timeout = timeout
        self.max_retries = max_retries
        self.session = requests.Session()
        self.session.headers.update(
            {
                "User-Agent": self.user_agent,
                "Accept-Encoding": "gzip, deflate",
                "Host": "www.sec.gov",
            }
        )
        # Separate headers for data.sec.gov.
        self.data_headers = {"User-Agent": self.user_agent, "Accept-Encoding": "gzip, deflate"}
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self._ticker_map = None

    # ------------------------------------------------------------------
    # HTTP
    # ------------------------------------------------------------------

    def _get(
        self, url: str, *, headers: Optional[dict] = None, stream: bool = False
    ) -> requests.Response:
        last_error = None
        for attempt in range(self.max_retries):
            try:
                time.sleep(self.request_delay)
                response = requests.get(
                    url,
                    headers=headers or self.session.headers,
                    timeout=self.timeout,
                    stream=stream,
                )
                if response.status_code == 200:
                    return response
                # Retry transient errors
                if response.status_code in {429, 500, 502, 503, 504}:
                    wait = 2**attempt
                    print(f"HTTP {response.status_code}. Retrying in {wait}s...")
                    time.sleep(wait)
                    last_error = RuntimeError(f"HTTP {response.status_code}: {url}")
                    continue
                response.raise_for_status()
            except requests.RequestException as exc:
                last_error = exc
                wait = 2**attempt
                print(f"Request failed: {exc}. Retrying in {wait}s...")
                time.sleep(wait)
        raise RuntimeError(
            f"Failed to download after {self.max_retries} attempts: {url}"
        ) from last_error

    # ------------------------------------------------------------------
    # Ticker -> CIK
    # ------------------------------------------------------------------

    def _load_ticker_map(self):
        if self._ticker_map is not None:
            return self._ticker_map
        url = "https://www.sec.gov/files/company_tickers.json"
        response = self._get(url, headers=self.data_headers)
        data = response.json()
        ticker_map = {}
        for item in data.values():
            ticker = item["ticker"].upper()
            ticker_map[ticker] = {
                "cik": str(item["cik_str"]).zfill(10),
                "title": item["title"],
            }
        self._ticker_map = ticker_map
        return ticker_map

    def get_company_info(self, ticker: str):
        ticker = ticker.upper()
        ticker_map = self._load_ticker_map()
        if ticker not in ticker_map:
            raise ValueError(f"Ticker '{ticker}' not found in SEC company ticker list.")
        return ticker_map[ticker]

    # ------------------------------------------------------------------
    # Submission metadata
    # ------------------------------------------------------------------

    def get_submissions(self, cik: str):
        cik = str(cik).zfill(10)
        url = f"{self.SEC_DATA_URL}/submissions/CIK{cik}.json"
        response = self._get(url, headers=self.data_headers)
        return response.json()

    # ------------------------------------------------------------------
    # Filing history
    # ------------------------------------------------------------------

    def _get_filing_history(self, submissions: dict, cik: str) -> list[dict]:
        recent = submissions.get("filings", {}).get("recent", {})
        records = []
        if recent:
            keys = ["accessionNumber", "filingDate", "reportDate", "form", "primaryDocument"]
            length = len(recent.get("form", []))
            for i in range(length):
                records.append({key: recent.get(key, [None] * length)[i] for key in keys})
        # SEC stores older filing history in separate JSON files.
        older_files = submissions.get("filings", {}).get("files", [])
        for file_info in older_files:
            name = file_info["name"]
            url = f"{self.SEC_DATA_URL}/submissions/{name}"
            response = self._get(url, headers=self.data_headers)
            older = response.json()
            length = len(older.get("form", []))
            for i in range(length):
                records.append(
                    {
                        "accessionNumber": older["accessionNumber"][i],
                        "filingDate": older["filingDate"][i],
                        "reportDate": older["reportDate"][i],
                        "form": older["form"][i],
                        "primaryDocument": older["primaryDocument"][i],
                    }
                )
        return records

    # ------------------------------------------------------------------
    # Date filtering
    # ------------------------------------------------------------------

    @staticmethod
    def _parse_date(value: Optional[str]) -> Optional[date]:
        if not value:
            return None
        return date.fromisoformat(value)

    def _in_date_range(
        self, filing_date: str, start_date: Optional[str], end_date: Optional[str]
    ) -> bool:
        filing = self._parse_date(filing_date)
        if filing is None:
            return False
        if start_date:
            if filing < self._parse_date(start_date):
                return False
        if end_date:
            if filing > self._parse_date(end_date):
                return False
        return True

    # ------------------------------------------------------------------
    # Build URLs
    # ------------------------------------------------------------------

    def _build_urls(self, cik: str, accession_number: str, primary_document: str):
        cik_number = str(int(cik))
        accession_no_dash = accession_number.replace("-", "")
        primary_url = (
            f"{self.SEC_ARCHIVES_URL}/"
            f"{cik_number}/"
            f"{accession_no_dash}/"
            f"{primary_document}"
        )
        filing_index_url = (
            f"{self.SEC_ARCHIVES_URL}/"
            f"{cik_number}/"
            f"{accession_no_dash}/"
            f"{accession_number}-index.html"
        )
        raw_submission_url = (
            f"{self.SEC_ARCHIVES_URL}/"
            f"{cik_number}/"
            f"{accession_no_dash}/"
            f"{accession_number}.txt"
        )
        return (primary_url, filing_index_url, raw_submission_url)

    # ------------------------------------------------------------------
    # HTML -> text
    # ------------------------------------------------------------------

    @staticmethod
    def html_to_text(html: str) -> str:
        soup = BeautifulSoup(html, "html.parser")
        # Remove elements that aren't useful for RAG.
        for tag in soup(["script", "style", "noscript", "svg"]):
            tag.decompose()
        text = soup.get_text(separator="\n")
        lines = []
        for line in text.splitlines():
            line = " ".join(line.split())
            if line:
                lines.append(line)
        return "\n".join(lines)

    # ------------------------------------------------------------------
    # Download one filing
    # ------------------------------------------------------------------

    def download_filing(
        self,
        filing: dict,
        ticker: str,
        company_name: str,
        cik: str,
        download_raw_txt: bool = True,
        overwrite: bool = False,
    ) -> FilingMetadata:
        accession = filing["accessionNumber"]
        primary_document = filing["primaryDocument"]
        filing_date = filing["filingDate"]
        report_date = filing["reportDate"]
        form = filing["form"]
        primary_url, filing_index_url, raw_submission_url = self._build_urls(
            cik, accession, primary_document
        )

        # --------------------------------------------------------------
        # Directory
        # --------------------------------------------------------------

        filing_dir = self.output_dir / ticker.upper() / form / report_date / accession
        filing_dir.mkdir(parents=True, exist_ok=True)
        html_path = filing_dir / primary_document
        txt_path = filing_dir / f"{Path(primary_document).stem}.txt"
        metadata_path = filing_dir / "metadata.json"

        # --------------------------------------------------------------
        # HTML
        # --------------------------------------------------------------

        if not html_path.exists() or overwrite:
            print(f"Downloading {ticker} {form} {report_date} ...")
            response = self._get(primary_url)
            html_path.write_bytes(response.content)
        else:
            print(f"Already exists: {html_path}")

        # --------------------------------------------------------------
        # Clean text
        # --------------------------------------------------------------

        if not txt_path.exists() or overwrite:
            html = html_path.read_text(encoding="utf-8", errors="ignore")
            clean_text = self.html_to_text(html)
            txt_path.write_text(clean_text, encoding="utf-8")

        # --------------------------------------------------------------
        # Optional raw submission
        # --------------------------------------------------------------

        raw_path = filing_dir / f"{accession}.txt"
        if download_raw_txt:
            if not raw_path.exists() or overwrite:
                response = self._get(raw_submission_url)
                raw_path.write_bytes(response.content)

        # --------------------------------------------------------------
        # Metadata
        # --------------------------------------------------------------

        metadata = FilingMetadata(
            ticker=ticker.upper(),
            company_name=company_name,
            cik=cik,
            form=form,
            filing_date=filing_date,
            report_date=report_date,
            accession_number=accession,
            primary_document=primary_document,
            primary_url=primary_url,
            filing_index_url=filing_index_url,
            raw_submission_url=raw_submission_url,
            local_html_path=str(html_path),
            local_txt_path=str(txt_path),
        )
        metadata_path.write_text(json.dumps(asdict(metadata), indent=2), encoding="utf-8")
        return metadata

    # ------------------------------------------------------------------
    # Get matching filings
    # ------------------------------------------------------------------

    def list_filings(
        self,
        ticker: str,
        forms: Iterable[str] = DEFAULT_FORMS,
        start_date: Optional[str] = None,
        end_date: Optional[str] = None,
    ) -> list[dict]:
        ticker = ticker.upper()
        company = self.get_company_info(ticker)
        cik = company["cik"]
        submissions = self.get_submissions(cik)
        records = self._get_filing_history(submissions, cik)
        forms = set(forms)
        results = []
        for filing in records:
            if filing["form"] not in forms:
                continue
            if not self._in_date_range(filing["filingDate"], start_date, end_date):
                continue
            results.append(filing)
        # Newest first
        results.sort(key=lambda x: x["filingDate"], reverse=True)
        return results

    # ------------------------------------------------------------------
    # Download company
    # ------------------------------------------------------------------

    def download_company(
        self,
        ticker: str,
        forms: Iterable[str] = DEFAULT_FORMS,
        start_date: Optional[str] = None,
        end_date: Optional[str] = None,
        download_raw_txt: bool = True,
        overwrite: bool = False,
    ) -> list[FilingMetadata]:
        ticker = ticker.upper()
        company = self.get_company_info(ticker)
        cik = company["cik"]
        company_name = company["title"]
        print()
        print("=" * 70)
        print(f"Company : {company_name}")
        print(f"Ticker  : {ticker}")
        print(f"CIK     : {cik}")
        print("=" * 70)
        print()
        filings = self.list_filings(
            ticker=ticker, forms=forms, start_date=start_date, end_date=end_date
        )
        print(f"Found {len(filings)} filings.")
        downloaded = []
        for filing in filings:
            metadata = self.download_filing(
                filing=filing,
                ticker=ticker,
                company_name=company_name,
                cik=cik,
                download_raw_txt=download_raw_txt,
                overwrite=overwrite,
            )
            downloaded.append(metadata)

        # --------------------------------------------------------------
        # Company-level manifest
        # --------------------------------------------------------------

        manifest_path = self.output_dir / ticker / "manifest.json"
        manifest_path.parent.mkdir(parents=True, exist_ok=True)
        manifest_path.write_text(
            json.dumps([asdict(x) for x in downloaded], indent=2), encoding="utf-8"
        )
        return downloaded

In [6]:
#from sec_downloader import SECDownloader
downloader = SECDownloader( user_agent="radha4bit@gmail.com", output_dir=r"D:\Fin_Rag\sec_filings", )


In [8]:
filings = downloader.download_company( ticker="AMD",
                                       forms=["10-K"],
                                       start_date="2020-01-01",
                                       end_date="2026-12-31",
                                     )


Company : ADVANCED MICRO DEVICES INC
Ticker  : AMD
CIK     : 0000002488

Found 7 filings.
